# 1. Dataset Construction and Cohort Definition

**Audience:** clinical/data science team.  
**Purpose:** document how raw follow-up data are converted into analysis cohorts, and make the right-censoring correction explicit.

**Key message:** the binary classification target is only valid for patients whose 7-year status is observable. Therefore, classification uses `strict` and `competing`; survival uses the full time-to-event cohort.

In [7]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures'
VENV_PY = ROOT / '.venv' / 'Scripts' / 'python.exe'
PIPELINE_PY = VENV_PY if VENV_PY.exists() else Path(sys.executable)
FEATURE_SETS = [
    'CV17', 'CV17_THY_CONT_STATES', 'CV17_THY_ABNORMAL_BIN', 'CV17_THY_STATE_ORD',
    'CV17_THY_CONT', 'CV17_THY_STATES', 'CV17_THY_CONT_STATES_RATIO', 'CV17_THY_CONT_RATIO'
]

def read_csv(name, folder=DATA, **kwargs):
    path = folder / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, **kwargs)

def show_image(path, width=850):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing figure: {path}')

def run_step(step, force=False):
    cmd = [str(PIPELINE_PY), str(ROOT / 'run_pipeline.py'), '--step', step]
    if force:
        cmd.append('--force')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

def pct(x):
    return f'{100*x:.1f}%'

## How to use this notebook

- Leave `RUN_PIPELINE_STEP = False` to review the current saved artefacts.
- Set it to `True` only when you intentionally want to rebuild cohorts from raw Excel files.
- The tables below are designed as audit evidence for the team.

In [8]:
RUN_PIPELINE_STEP = True
if RUN_PIPELINE_STEP:
    run_step('build', force=False)
else:
    print('Using existing processed cohorts. Set RUN_PIPELINE_STEP=True to rebuild with .venv.')

Running: c:\Users\ileni\ml4cad\.venv\Scripts\python.exe c:\Users\ileni\ml4cad\run_pipeline.py --step build


## 1.1 Cohorts produced

This table is the first sanity check: sample sizes and event counts must match the expected design.

In [9]:
full = read_csv('cohort_full.csv')
strict = read_csv('cohort_strict.csv')
competing = read_csv('cohort_competing.csv')
survival = read_csv('cohort_survival.csv')

summary = pd.DataFrame([
    {'cohort': 'full', 'purpose': 'raw merged cohort for audit', 'n': len(full), 'events_cvd': int(full['event_cvd'].sum()), 'y7_prevalence': np.nan},
    {'cohort': 'strict', 'purpose': 'primary binary classification', 'n': len(strict), 'events_cvd/y7=1': int(strict['y7'].sum()), 'y7_prevalence': strict['y7'].mean()},
    {'cohort': 'competing', 'purpose': 'sensitivity binary classification', 'n': len(competing), 'events_cvd/y7=1': int(competing['y7'].sum()), 'y7_prevalence': competing['y7'].mean()},
    {'cohort': 'survival', 'purpose': 'full survival analysis', 'n': len(survival), 'events_cvd': int(survival['event_cvd'].sum()), 'y7_prevalence': np.nan},
])
display(summary)

,cohort,purpose,n,events_cvd,y7_prevalence,events_cvd/y7=1
0,full,raw merged cohort for audit,8065,1008.0,NaN,NaN
1,strict,primary binary classification,4390,NaN,0.192027,843.0
2,competing,sensitivity binary classification,5437,NaN,0.155049,843.0
3,survival,full survival analysis,8063,1008.0,NaN,NaN


## 1.2 Target audit: why `strict` is valid

Patients with unknown 7-year status are **not** labelled as survivors. This is the central methodological correction.

In [10]:
HORIZON_DAYS = 7 * 365.25
cvd_within = (full['event_cvd'].eq(1) & full['time_days'].le(HORIZON_DAYS))
event_free = (full['time_days'].ge(HORIZON_DAYS) & ~cvd_within)
noncvd_early = (full['event_noncvd'].eq(1) & full['time_days'].lt(HORIZON_DAYS))
alive_short = (full['event_death'].eq(0) & full['time_days'].lt(HORIZON_DAYS))
strict_expected = set(full.loc[cvd_within | event_free, 'Number'])
competing_expected = set(full.loc[cvd_within | event_free | noncvd_early, 'Number'])

audit = pd.DataFrame([
    {'quantity': 'CVD death <=7y -> y7=1', 'n': int(cvd_within.sum()), 'interpretation': 'positive class'},
    {'quantity': 'Event-free observed >=7y -> y7=0', 'n': int(event_free.sum()), 'interpretation': 'negative class in strict'},
    {'quantity': 'Alive with FU <7y', 'n': int(alive_short.sum()), 'interpretation': 'removed from classification'},
    {'quantity': 'Non-CVD death <7y', 'n': int(noncvd_early.sum()), 'interpretation': 'removed from strict; negative in competing'},
    {'quantity': 'Strict membership OK', 'n': strict_expected == set(strict['Number']), 'interpretation': 'must be True'},
    {'quantity': 'Competing membership OK', 'n': competing_expected == set(competing['Number']), 'interpretation': 'must be True'},
])
display(audit)

,quantity,n,interpretation
0,CVD death <=7y -> y7=1,843,positive class
1,Event-free observed >=7y -> y7=0,3547,negative class in strict
2,Alive with FU <7y,2628,removed from classification
3,Non-CVD death <7y,1047,removed from strict; negative in competing
4,Strict membership OK,True,must be True
5,Competing membership OK,True,must be True


## 1.3 Thyroid state and data-quality corrections

The thyroid state variables should be mutually exclusive after correction. The correction log documents patient-level edits without modifying raw files.

In [11]:
thyroid_cols = ['Euthyroid','SCH','SCT','Low_T3','Hypothyroid','Hyperthyroid']
rows = []
for name, df in [('full', full), ('strict', strict), ('competing', competing)]:
    state_sum = df[thyroid_cols].sum(axis=1)
    rows.append({'cohort': name, 'all_exactly_one_state': bool((state_sum == 1).all()), 'n': len(df)})
state_audit = pd.DataFrame(rows)
display(state_audit)

display(strict[thyroid_cols].sum().rename('strict_n').reset_index().rename(columns={'index':'thyroid_state'}))

path = REPORTS / 'data_quality_corrections.csv'
if path.exists():
    display(pd.read_csv(path))
else:
    print('No data_quality_corrections.csv found.')

,cohort,all_exactly_one_state,n
0,full,True,8065
1,strict,True,4390
2,competing,True,5437


,thyroid_state,strict_n
0,Euthyroid,2902.0
1,SCH,261.0
2,SCT,155.0
3,Low_T3,910.0
4,Hypothyroid,75.0
5,Hyperthyroid,87.0


,Number,issue,status,before,updates,after,rationale
0,6850,Conflicting thyroid flags: SCH=1 and Hyperthyr...,corrected,"{'Number': 6850.0, 'TSH': 4.76, 'fT3': 23.5, '...","{'SCH': 1, 'Hyperthyroid': 0}","{'Number': 6850.0, 'TSH': 4.76, 'fT3': 23.5, '...",TSH=4.76 is elevated and is more coherent with...
1,7286,Death date is present but Total mortality=0,corrected,"{'Number': 7286, 'Data prelievo': Timestamp('2...",{'Total mortality': 1},"{'Number': 7286, 'Data prelievo': Timestamp('2...",A populated death date should be consistent wi...


## 1.4 Feature sets

These are the feature definitions used throughout classification and survival comparisons.

In [12]:
from configs.config import FEATURE_SETS as PROJECT_FEATURE_SETS
fs_table = pd.DataFrame([
    {'feature_set': k, 'n_features': len(v), 'features': ', '.join(v)}
    for k, v in PROJECT_FEATURE_SETS.items()
])
display(fs_table)

,feature_set,n_features,features
0,CV17,17,"Gender, Age, Angina, Previous_CABG, Previous_P..."
1,CV17_THY_CONT_STATES,26,"Gender, Age, Angina, Previous_CABG, Previous_P..."
2,CV17_THY_ABNORMAL_BIN,18,"Gender, Age, Angina, Previous_CABG, Previous_P..."
3,CV17_THY_STATE_ORD,18,"Gender, Age, Angina, Previous_CABG, Previous_P..."
4,CV17_THY_CONT,20,"Gender, Age, Angina, Previous_CABG, Previous_P..."
5,CV17_THY_STATES,23,"Gender, Age, Angina, Previous_CABG, Previous_P..."
6,CV17_THY_CONT_STATES_RATIO,27,"Gender, Age, Angina, Previous_CABG, Previous_P..."
7,CV17_THY_CONT_RATIO,21,"Gender, Age, Angina, Previous_CABG, Previous_P..."


## Take-home message

- `strict` is the primary classification cohort because every patient has an observed 7-year status.
- `competing` is a sensitivity cohort that treats early non-CVD deaths as non-CVD events.
- `full/survival` remains the correct cohort for absolute time-to-event risk estimation.